# Student Dropout Prediction using Decision Tree Classifier

This notebook implements a decision tree classifier to predict student dropout using stratified 10-fold cross-validation. The approach focuses on experimentation and baseline model performance without hyperparameter tuning.

## Table of Contents

1. **[Import Libraries & Load Data](#1-import-libraries--load-data)** - Load necessary Python packages and dataset
2. **[Prepare Data](#2-prepare-data)** - Split data and prepare features and target variables
3. **[Cross-Validation Training](#3-stratified-10-fold-cross-validation-training-and-evaluation)** - Train and evaluate model using 10-fold CV
4. **[CV Results Analysis](#4-cross-validation-results-analysis)** - Analyze cross-validation performance metrics
5. **[Final Model Training](#5-train-final-model-and-validation-analysis)** - Train final model and perform holdout validation
6. **[ROC Curve Analysis](#6-roc-curve-analysis)** - Visualize model discriminative performance
7. **[Feature Importance Analysis](#7-feature-importance-analysis)** - Examine feature importance and decision tree structure
8. **[Test Set Evaluation](#8-final-evaluation-on-test-set)** - Evaluate final model on held-out test data
9. **[Comprehensive Analysis](#9-comprehensive-model-evaluation-and-analysis)** - Complete performance analysis and recommendations

## Key Insights
- **Baseline Accuracy**: 68.9% (majority class prediction)
- **Cross-Validation Performance**: To be determined through 10-fold CV
- **Model Type**: Decision Tree with balanced class weights
- **Evaluation Strategy**: Stratified 10-fold cross-validation for robust assessment
- **Advantages**: Interpretable, handles non-linear patterns, no feature scaling required

## 1. Import Libraries & Load Data

In [ ]:
# Import required libraries
import os
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.tree import DecisionTreeClassifier
from sklearn.tree import plot_tree, export_text
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_auc_score, roc_curve
from sklearn.model_selection import cross_validate, StratifiedKFold, train_test_split
from sklearn.metrics import make_scorer
import time
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('default')
sns.set_palette("husl")

## 2. Load Dataset

In [ ]:
# Load the dataset
current_user = os.getlogin()
PROCESSED_DIR = Path(f"/home/{current_user}/local-share/processed")
df = pd.read_csv(PROCESSED_DIR / 'feature_selection.csv')

print("Dataset shape:", df.shape)
print("\nDataset info:")
print(df.info())

## 3. Prepare Data

**Purpose**: Split data into train/validation/test, define target feature and prepare all features.

In [ ]:
# Split data based on the 'set' column
train_df = df[df['set'] == 'train'].copy()
val_df = df[df['set'] == 'val'].copy()
test_df = df[df['set'] == 'test'].copy()

print(f"Training set shape: {train_df.shape}")
print(f"Validation set shape: {val_df.shape}")
print(f"Test set shape: {test_df.shape}")

In [ ]:
# Specify the target variable
target_col = 'sdo5_degree_drop_out'

# Exclude non-feature columns (set, target, any IDs)
exclude_cols = ['set', target_col]

# Get feature columns
feature_cols = [col for col in df.columns if col not in exclude_cols]
print(f"Number of features: {len(feature_cols)}")
print(f"Feature columns: {feature_cols}")

# Prepare training data
X_train = train_df[feature_cols]
y_train = train_df[target_col]

# Prepare validation data
X_val = val_df[feature_cols]
y_val = val_df[target_col]

# Prepare test data
X_test = test_df[feature_cols]
y_test = test_df[target_col]

## 4. Stratified 10-Fold Cross-Validation Training and Evaluation

**Purpose**: Train and evaluate decision tree using stratified 10-fold cross-validation.

**Methodology**:
- Combine train+validation sets for CV (test set remains separate)
- Stratified K-fold maintains class distribution across folds
- Multiple metrics: accuracy, ROC AUC, precision, recall, F1-score
- Class balancing with `class_weight='balanced'`
- Default hyperparameters for baseline performance assessment

In [ ]:
# Combine training and validation data for cross-validation
# (Keep test set separate for final evaluation)
X_train_val = pd.concat([X_train, X_val], axis=0)
y_train_val = pd.concat([y_train, y_val], axis=0)

print(f"\nCombined training + validation set shape: {X_train_val.shape}")
print(f"Target distribution in combined set:")
print(y_train_val.value_counts(normalize=True))

# Set up stratified 10-fold cross-validation
cv_strategy = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# Define scoring metrics for cross-validation
scoring_metrics = {
    'accuracy': 'accuracy',
    'roc_auc': 'roc_auc',
    'precision': 'precision',
    'recall': 'recall',
    'f1': 'f1'
}

# Initialize the decision tree model
cv_model = DecisionTreeClassifier(
    random_state=42,
    class_weight='balanced'
)

print("\nStarting stratified 10-fold cross-validation...")
print("This will train and evaluate 10 different models...")

start_time = time.time()

# Perform cross-validation
cv_results = cross_validate(
    cv_model,
    X_train_val,
    y_train_val,
    cv=cv_strategy,
    scoring=scoring_metrics,
    return_train_score=True,
    n_jobs=-1
)

end_time = time.time()
print(f"Cross-validation completed in {end_time - start_time:.2f} seconds")

## 5. Cross-Validation Results Analysis

**Purpose**: Analyze and interpret 10-fold cross-validation performance metrics.

**Analysis**:
- Statistical summary: mean, std, min, max, median for each metric
- Performance consistency across folds
- Key metrics extraction for downstream analysis
- Confidence assessment through variance analysis

In [ ]:
# Analyze and visualize cross-validation results
from scipy import stats

print("=" * 80)
print("10-FOLD CROSS-VALIDATION RESULTS")
print("=" * 80)

# Extract test scores (what we care about for model evaluation)
test_scores = {
    'Accuracy': cv_results['test_accuracy'],
    'ROC AUC': cv_results['test_roc_auc'],
    'Precision': cv_results['test_precision'],
    'Recall': cv_results['test_recall'],
    'F1-Score': cv_results['test_f1']
}

# Calculate statistics for each metric
cv_stats = {}
for metric, scores in test_scores.items():
    cv_stats[metric] = {
        'mean': np.mean(scores),
        'std': np.std(scores),
        'min': np.min(scores),
        'max': np.max(scores),
        'median': np.median(scores)
    }

# Display summary statistics
print(f"\n📊 CROSS-VALIDATION PERFORMANCE SUMMARY:")
print(f"{'Metric':<12} {'Mean':<8} {'Std':<8} {'Min':<8} {'Max':<8} {'Median':<8}")
print("-" * 60)
for metric, stats in cv_stats.items():
    print(f"{metric:<12} {stats['mean']:.3f}   {stats['std']:.3f}   {stats['min']:.3f}   {stats['max']:.3f}   {stats['median']:.3f}")

# Store main CV results for later use
cv_mean_accuracy = cv_stats['Accuracy']['mean']
cv_mean_roc_auc = cv_stats['ROC AUC']['mean']

print(f"\n🎯 KEY METRICS:")
print(f"   • Mean CV Accuracy: {cv_mean_accuracy:.3f} ± {cv_stats['Accuracy']['std']:.3f}")
print(f"   • Mean CV ROC AUC: {cv_mean_roc_auc:.3f} ± {cv_stats['ROC AUC']['std']:.3f}")

print("=" * 80)

## 6. Train Final Model and Validation Analysis

**Purpose**: Train final model on full train+validation set and perform holdout validation.

**Process**:
- Train final model on combined train+validation data
- Create holdout subset for visualization and comparison
- Generate confusion matrix and classification metrics
- Compare holdout results with CV performance for consistency

In [ ]:
# Train final model on full training+validation set for feature analysis and final evaluation
print("Training final model on combined training+validation data...")

dt_model = DecisionTreeClassifier(
    random_state=42,
    class_weight='balanced'
)

# Fit on the combined train+val set
dt_model.fit(X_train_val, y_train_val)

print(f"Final model trained successfully!")
print(f"Number of features used: {dt_model.n_features_in_}")
print(f"Tree depth: {dt_model.get_depth()}")
print(f"Number of leaves: {dt_model.get_n_leaves()}")
print(f"Number of nodes: {dt_model.tree_.node_count}")

# For comparison with CV results, let's create a validation subset prediction
# Using a simple holdout for visualization purposes
X_temp, X_val_holdout, y_temp, y_val_holdout = train_test_split(
    X_train_val, y_train_val, test_size=0.2, random_state=42, stratify=y_train_val
)

# Train on temp set, predict on holdout
dt_temp = DecisionTreeClassifier(random_state=42, class_weight='balanced')
dt_temp.fit(X_temp, y_temp)

y_val_pred = dt_temp.predict(X_val_holdout)
y_val_pred_proba = dt_temp.predict_proba(X_val_holdout)[:, 1]

# Calculate validation metrics for comparison
val_accuracy = accuracy_score(y_val_holdout, y_val_pred)
val_roc_auc = roc_auc_score(y_val_holdout, y_val_pred_proba)

print(f"\n=== Holdout Validation Performance (for comparison) ===")
print(f"Holdout Accuracy: {val_accuracy:.4f} (CV Mean: {cv_mean_accuracy:.4f})")
print(f"Holdout ROC AUC: {val_roc_auc:.4f} (CV Mean: {cv_mean_roc_auc:.4f})")

print(f"\n=== Classification Report (Holdout) ===")
print(classification_report(y_val_holdout, y_val_pred))

# Confusion Matrix for holdout
print(f"\n=== Confusion Matrix (Holdout) ===")
cm = confusion_matrix(y_val_holdout, y_val_pred)
print(cm)

# Visualize confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.title('Confusion Matrix - Holdout Validation Set')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

## 7. ROC Curve Analysis

**Purpose**: Visualize model's discriminative performance using ROC curve.

**Analysis**:
- Plot ROC curve for holdout validation set
- Compare with random baseline (diagonal line)
- Display AUC score for quantitative assessment
- Compare holdout AUC with cross-validation mean for consistency

In [ ]:
# Plot ROC Curve for holdout validation
fpr, tpr, thresholds = roc_curve(y_val_holdout, y_val_pred_proba)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {val_roc_auc:.4f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random baseline')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - Decision Tree Model (Holdout Validation)')
plt.legend(loc="lower right")
plt.grid(True, alpha=0.3)
plt.show()

# Add note about CV vs holdout
print(f"Note: This ROC curve is based on a single holdout validation.")
print(f"The cross-validation results provide a more robust estimate:")
print(f"CV Mean ROC AUC: {cv_mean_roc_auc:.4f} ± {cv_stats['ROC AUC']['std']:.4f}")

## 8. Feature Importance Analysis

**Purpose**: Examine decision tree feature importance to understand which variables drive predictions.

**Analysis**:
- Extract and rank feature importance values from the trained tree
- Visualize top 20 most important features
- Interpret feature importance in context of decision tree splits
- Display tree structure insights and decision rules

In [ ]:
# Get feature importances from the trained model
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': dt_model.feature_importances_
}).sort_values('importance', ascending=False)

print("Feature Importance Rankings:")
print(feature_importance)

# Plot feature importances
plt.figure(figsize=(12, 8))
top_features = feature_importance.head(20)  # Show top 20 features
plt.barh(range(len(top_features)), top_features['importance'], color='forestgreen', alpha=0.7)
plt.yticks(range(len(top_features)), top_features['feature'])
plt.xlabel('Feature Importance')
plt.title('Top 20 Feature Importances - Decision Tree Model')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

# Display top 10 most important features
print(f"\nTop 10 most important features:")
for i, row in feature_importance.head(10).iterrows():
    print(f"{row['feature']}: {row['importance']:.4f}")

# Display tree structure information
print(f"\n=== Decision Tree Structure ===")
print(f"Tree depth: {dt_model.get_depth()}")
print(f"Number of leaves: {dt_model.get_n_leaves()}")
print(f"Number of nodes: {dt_model.tree_.node_count}")
print(f"Number of features used: {np.sum(dt_model.feature_importances_ > 0)}")

# Show decision tree rules (first few levels only for readability)
tree_rules = export_text(dt_model, feature_names=feature_cols, max_depth=3)
print(f"\n=== Decision Tree Rules (first 3 levels) ===")
print(tree_rules)

## 9. Final Evaluation on Test Set

**Purpose**: Evaluate final model performance on completely unseen test data.

In [ ]:
# Use the original decision tree model for final evaluation
final_model = dt_model
print("Using original decision tree model for final evaluation")

# Make predictions on test set
y_test_pred = final_model.predict(X_test)
y_test_pred_proba = final_model.predict_proba(X_test)[:, 1]

# Calculate test metrics
test_accuracy = accuracy_score(y_test, y_test_pred)
test_roc_auc = roc_auc_score(y_test, y_test_pred_proba)

print("\n=== Final Test Set Performance ===")
print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"Test ROC AUC Score: {test_roc_auc:.4f}")

print("\n=== Test Set Classification Report ===")
print(classification_report(y_test, y_test_pred))

# Test set confusion matrix
print("\n=== Test Set Confusion Matrix ===")
test_cm = confusion_matrix(y_test, y_test_pred)
print(test_cm)

# Visualize test confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(test_cm, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.title('Confusion Matrix - Test Set')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

# Summary of model performance across all sets
print("\n=== Model Performance Summary ===")
print(f"Cross-Validation Accuracy: {cv_mean_accuracy:.4f} ± {cv_stats['Accuracy']['std']:.4f}")
print(f"Cross-Validation ROC AUC: {cv_mean_roc_auc:.4f} ± {cv_stats['ROC AUC']['std']:.4f}")
print(f"Holdout Validation Accuracy: {val_accuracy:.4f}")
print(f"Holdout Validation ROC AUC: {val_roc_auc:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"Test ROC AUC: {test_roc_auc:.4f}")

## 10. Comprehensive Model Evaluation and Analysis

**Purpose**: Provide complete performance analysis, business impact assessment, and recommendations.

In [ ]:
# Comprehensive Model Evaluation Analysis

print("=" * 80)
print("COMPREHENSIVE MODEL EVALUATION ANALYSIS")
print("=" * 80)

# Calculate baseline accuracy (predicting majority class)
test_dropout_rate = y_test.mean()
baseline_accuracy = max(test_dropout_rate, 1 - test_dropout_rate)

print(f"\n📊 DATASET CHARACTERISTICS:")
print(f"   • Test set size: {len(y_test):,} students")
print(f"   • Dropout rate: {test_dropout_rate:.1%}")
print(f"   • Non-dropout rate: {1-test_dropout_rate:.1%}")
print(f"   • Baseline accuracy (majority class): {baseline_accuracy:.1%}")

print(f"\n🎯 MODEL PERFORMANCE vs BASELINE:")
print(f"   • Test Accuracy: {test_accuracy:.1%} (vs {baseline_accuracy:.1%} baseline)")
print(f"   • Accuracy improvement: {test_accuracy - baseline_accuracy:+.1%}")
print(f"   • ROC AUC: {test_roc_auc:.3f} (0.5 = random, 1.0 = perfect)")

# Performance analysis
if test_accuracy > baseline_accuracy:
    print(f"   ✅ Model beats baseline by {test_accuracy - baseline_accuracy:.1%}")
else:
    print(f"   ❌ Model underperforms baseline by {baseline_accuracy - test_accuracy:.1%}")

# ROC AUC interpretation
if test_roc_auc > 0.8:
    auc_rating = "Excellent"
elif test_roc_auc > 0.7:
    auc_rating = "Good"
elif test_roc_auc > 0.6:
    auc_rating = "Fair"
else:
    auc_rating = "Poor"

print(f"   📈 AUC Rating: {auc_rating}")

# Confusion Matrix Analysis
tn, fp, fn, tp = test_cm.ravel()
sensitivity = tp / (tp + fn)  # True Positive Rate (Recall)
specificity = tn / (tn + fp)  # True Negative Rate
precision = tp / (tp + fp)    # Positive Predictive Value
npv = tn / (tn + fn)         # Negative Predictive Value

print(f"\n🔍 DETAILED PERFORMANCE METRICS:")
print(f"   • Sensitivity (Recall): {sensitivity:.1%} - Correctly identified dropouts")
print(f"   • Specificity: {specificity:.1%} - Correctly identified non-dropouts")
print(f"   • Precision: {precision:.1%} - Accuracy of dropout predictions")
print(f"   • Negative Predictive Value: {npv:.1%} - Accuracy of non-dropout predictions")

print(f"\n🎭 CONFUSION MATRIX BREAKDOWN:")
print(f"   • True Negatives: {tn:,} (Correctly predicted non-dropouts)")
print(f"   • False Positives: {fp:,} (Incorrectly predicted as dropouts)")
print(f"   • False Negatives: {fn:,} (Missed dropouts)")
print(f"   • True Positives: {tp:,} (Correctly predicted dropouts)")

# Business Impact Analysis
missed_dropouts_rate = fn / (tp + fn)
false_alarm_rate = fp / (tn + fp)

print(f"\n💼 BUSINESS IMPACT:")
print(f"   • Missed dropout rate: {missed_dropouts_rate:.1%} ({fn:,} students)")
print(f"   • False alarm rate: {false_alarm_rate:.1%} ({fp:,} students)")

print(f"\n🌳 MODEL CONFIGURATION:")
print(f"   • Model type: Decision Tree")
print(f"   • Parameters: Default with class_weight='balanced'")
print(f"   • Tree depth: {final_model.get_depth()}")
print(f"   • Number of leaves: {final_model.get_n_leaves()}")
print(f"   • Number of nodes: {final_model.tree_.node_count}")
print(f"   • Features used: {np.sum(final_model.feature_importances_ > 0)}/{len(feature_cols)}")

print(f"\n🎯 RECOMMENDATIONS:")
if test_accuracy < baseline_accuracy:
    print("   ❗ Model performance is below baseline - consider:")
    print("     - Hyperparameter tuning (max_depth, min_samples_split, etc.)")
    print("     - Ensemble methods (Random Forest, Gradient Boosting)")
    print("     - Feature engineering and selection")
    print("     - More data collection")
elif test_roc_auc < 0.7:
    print("   ⚠️  Model shows limited predictive power - consider:")
    print("     - Ensemble methods (Random Forest, XGBoost)")
    print("     - Pruning to reduce overfitting")
    print("     - Feature selection/engineering")
    print("     - Hyperparameter optimization")
else:
    print("   ✅ Model shows reasonable performance but could be improved:")
    print("     - Ensemble methods for better accuracy and robustness")
    print("     - Hyperparameter tuning for optimal tree structure")
    print("     - Feature interaction exploration")
    print("     - Threshold optimization for business needs")
    print("     - Cross-validation hyperparameter search")

print(f"\n🌳 DECISION TREE INSIGHTS:")
print("   • Interpretable model - can trace exact decision paths")
print("   • Handles non-linear relationships naturally")
print("   • No need for feature scaling")
print("   • Prone to overfitting - single tree can memorize training data")
print("   • Feature importance based on how well features separate classes")
print("   • This experiment uses default parameters for baseline assessment")

# Cross-validation vs test performance comparison
cv_test_diff = test_accuracy - cv_mean_accuracy
print(f"\n📊 CROSS-VALIDATION vs TEST PERFORMANCE:")
print(f"   • CV Mean Accuracy: {cv_mean_accuracy:.4f} ± {cv_stats['Accuracy']['std']:.4f}")
print(f"   • Test Accuracy: {test_accuracy:.4f}")
print(f"   • Difference: {cv_test_diff:+.4f}")

if abs(cv_test_diff) > 0.05:
    if cv_test_diff > 0:
        consistency_rating = "⚠️  Test performance unexpectedly higher than CV - possible data leakage?"
    else:
        consistency_rating = "⚠️  Test performance lower than CV - possible overfitting"
else:
    consistency_rating = "✅ Good consistency between CV and test performance"

print(f"   • Consistency: {consistency_rating}")

print("\n" + "=" * 80)